<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Judgments from a language model

**[Judgments from a language model](https://learning.nextia-ai.com/courses/search/m02/judgments-from-a-language-model/)** · Search and Ranking: Find the Best Match · Module 2, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compare a language model's recorded relevance labels with people's labels, find which of its grades you can trust, and see how its labels for unjudged results change a measurement.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells (about 3 MB). No account, no key and no model download. |
| **You should know** | Cohen's kappa, from [Relevance judgments](https://learning.nextia-ai.com/courses/search/m02/relevance-judgments/), and NDCG, from [Ranking measures](https://learning.nextia-ai.com/courses/search/m02/ranking-measures/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m02/judgments-from-a-language-model/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M02-L04-judgments-from-a-language-model/judgments-from-a-language-model-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 2 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/recorded/llm_judgments.jsonl": "f1c7208b12acd187eb46e57def8a0eef3fb7de907dd9620b9000d2ecfce86624",
    "data/recorded/rerank.csv.gz": "38971117046aefd883c159ed9a7b196a0ca88f54757a8dc35df956d39222da88",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The model's labels and the people's

The course team asked a small chat model to grade 750 query–product pairs with the course's judging guidelines. Its answers are **recorded**: this notebook reads them, so you need no key. 450 pairs were already judged by the WANDS judges (150 exact, 150 partial, 150 irrelevant). The other 300 were in the top 10 of a search but had no judgment. Run the next cell.

In [ ]:
llm = pd.read_json("data/recorded/llm_judgments.jsonl", lines=True)
print(llm["why_sampled"].value_counts())
print("cost of all", len(llm), "labels: US$", round(llm["usd"].sum(), 3))
both = llm[llm["human_grade"] >= 0]           # pairs that the WANDS judges also graded
names = {2: "exact", 1: "partial", 0: "irrelevant"}
order = ["exact", "partial", "irrelevant"]
table = pd.crosstab(both["human_grade"].map(names).rename("WANDS judges"),
                    both["llm_grade"].map(names).rename("model")).loc[order, order]
table

> **Check.** All 750 labels cost about US$0.028. The rows are the WANDS grades, the columns the model's grades. The diagonal (95, 85, 129) is where they agree.

## 2. Generous or strict?

> **Predict.** When the model and the judges disagree, does the model give the higher grade or the lower one?

In [ ]:
higher = (both["llm_grade"] > both["human_grade"]).sum()
lower = (both["llm_grade"] < both["human_grade"]).sum()
one_step = ((both["llm_grade"] - both["human_grade"]).abs() == 1).sum()
print(f"model higher on {higher} pairs, lower on {lower}; {one_step} of the {higher + lower} disagreements are one grade apart")

> **Check.** The model is higher on 28 pairs and lower on 113: it is the **strict** judge. Most disagreements (136) are only one grade apart, which is why the weighted kappa (0.63) is higher than the plain kappa (0.53).

## 3. Your turn: which of its grades can you trust?

Agreement is one number for the whole table. For a decision, ask a sharper question: *when the model says "exact", how often do the people agree?* Complete `trust`.

In [ ]:
def trust(label):
    """When the model gives this grade, the share of pairs where the WANDS judges gave the same grade."""
    said = both[both["llm_grade"] == label]
    return (said["human_grade"] == label).mean()

trust_exact = trust(2)
trust_partial = trust(1)
trust_irrelevant = trust(0)
print(f"model says exact: right {trust_exact:.3f}; partial: {trust_partial:.3f}; irrelevant: {trust_irrelevant:.3f}")

In [ ]:
expect('trust_exact', trust_exact, 0.931, tolerance=0.001)

> **Check.** When the model says exact, the people agree 0.931 of the time. Its "partial" (0.545) and "irrelevant" (0.672) are much weaker: the line between partial and irrelevant is where the model and the people disagree most. Remember that the pairs were 150 per grade; on real search results, where most judged products are partial, these shares would change.

## 4. Fill the holes in the judgments

An unjudged product counts as irrelevant. A search that finds products that nobody judged looks worse than it is. The model labelled a sample of 300 unjudged results. Put its labels in the holes and measure hybrid search again on the English and French test queries.

In [ ]:
q = queries()
j = judgments()
rr = pd.read_csv("data/recorded/rerank.csv.gz")
test = q[(q["split"] == "test") & q["source"].isin(["wands", "translated"])]      # English and French WANDS queries
run = {qid: list(g.sort_values("hybrid_rank")["product_id"]) for qid, g in rr[rr["query_id"].isin(set(test["query_id"]))].groupby("query_id")}

filled = {qid: dict(v) for qid, v in j.items()}
unjudged = llm[llm["human_grade"] < 0]
for r in unjudged.itertuples():
    filled.setdefault(r.query_id, {})[r.product_id] = int(r.llm_grade)

people = cp.per_query(run, j)
with_model = cp.per_query(run, filled)
print(f"{len(run)} queries; judged in the top 10: {people['judged@10'].mean():.3f}")
print(f"NDCG@10 with unjudged = irrelevant: {people['ndcg@10'].mean():.3f}; with the model's labels: {with_model['ndcg@10'].mean():.3f}")

> **Check.** Only 0.715 of the top-10 results were judged. With the model's labels, NDCG@10 moves from 0.585 to 0.602. The method did not change: only the judgments did. Report both numbers, and say which judgments each one uses.

## 5. In the project

```sh
search judge llm      # the recorded model labels against the WANDS judges: agreement, kappa and the table
```

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The inverted index](https://learning.nextia-ai.com/courses/search/m03/the-inverted-index/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m02/judgments-from-a-language-model/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.